In [1]:
#행렬과 관련된 수학식을 사용할 수 있는 라이브러리
import numpy as np

#데이터분석용 pandas 사용을 위한 라이브러리
import pandas as pd

#그래프, 시각화를 위한 라이브러리
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

#문자데이터를 숫자로 인코딩하기 위한 라이브러리
from sklearn.preprocessing import LabelEncoder

#전체 데이터를 학습용 데이터와 검증용 데이터로 나누기 위한 라이브러리
from sklearn.model_selection import train_test_split

#분석 평가 지표 사용을 위한 라이브러리
from sklearn.metrics import  mean_absolute_error, mean_squared_error, r2_score

In [2]:
fm._load_fontmanager(try_read_cache=False)
plt.rc("font", family="NanumGothic")
plt.rc("axes", unicode_minus=False)

In [3]:
df = pd.read_csv('가스_인구_기온_통합.csv',  encoding='cp949')
df

,연월,시도,월평균기온,월평균최고기온,월평균최저기온,가구수(개),공급량(㎥),총인구수
0,2021-01,강원,-4.2,1.5,-9.7,386599,51918.194190,1541696
1,2021-02,강원,1.3,7.3,-4.6,387774,43068.962620,1540875
2,2021-03,강원,7.0,13.3,1.3,388403,32782.803000,1536399
3,2021-04,강원,11.8,17.9,5.7,390610,20762.350470,1536175
4,2021-05,강원,15.3,21.1,10.0,391098,14472.110240,1535491
...,...,...,...,...,...,...,...,...
1183,2026-02,충북,1.1,7.7,-5.0,584666,47598.860310,1596816
1184,2026-03,충북,6.8,13.6,0.5,579452,39099.270910,1597976
1185,2026-04,충북,13.5,20.6,6.4,576171,21893.604080,1599375
1186,2026-05,충북,18.6,25.5,11.8,576065,12157.524940,1600216


In [5]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

#랜덤포레스트 객체 생성
rf_model = RandomForestRegressor(
  n_estimators= 100, #트리를 몇 개 만들지 지정
  random_state= 42,
  n_jobs = -1
)

#xg_boost 객체 생성
xg_model = XGBRegressor(
  n_estimators=100,
  learning_rate=0.1,
  random_state=42,
  n_jobs=-1
)

result_model = []
result_anls = []
train_col = ['월평균기온']

for region in df['시도'].unique():
    sub = df[df['시도'] == region]

    #학습데이터와 검증데이터를 분리
    train_df, test_df = train_test_split(sub, test_size=0.2, random_state=42)
    X_train = train_df[train_col]

    #학습에 대한 답 데이터(train_df의 발전량 데이터)
    y_train = train_df['공급량(㎥)']

    #검증용 데이터
    X_test = test_df[train_col]
    y_test = test_df['공급량(㎥)']

    #랜덤포레스트 학습
    rf_model.fit(X_train, y_train)

    #예측
    rf_predict = rf_model.predict(X_test)

    #XGBoost 학습
    xg_model.fit(X_train, y_train)

    #예측
    xg_predict = xg_model.predict(X_test)

    print(f"[{region}] RF 학습완료, XGBoost 학습완료")

    #랜덤포레스트 결과 분석
    rf_mae = mean_absolute_error(y_test, rf_predict)
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predict))
    rf_r2 = r2_score(y_test, rf_predict)

    #xg부스터 결과 분석
    xg_mae = mean_absolute_error(y_test, xg_predict)
    xg_rmse = np.sqrt(mean_squared_error(y_test, xg_predict))
    xg_r2 = r2_score(y_test, xg_predict)

    print(f'랜덤포레스트 MAE = {rf_mae}, RMSE = {rf_rmse}, R2 = {rf_r2}')
    print(f'XGBOOST MAE = {xg_mae}, RMSE = {xg_rmse}, R2 = {xg_r2}')

    result_anls.append({'region': region, 
                         'rf_mae': rf_mae, 'rf_rmse': rf_rmse, 'rf_r2' : rf_r2,
                         'xg_mae': xg_mae, 'xg_rmse': xg_rmse, 'xg_r2' : xg_r2})

    #공급량의 값이 계절에 따라 급락이 큼 오차가 크게 나옴

[강원] RF 학습완료, XGBoost 학습완료
랜덤포레스트 MAE = 3961.823500512689, RMSE = 5415.055784047702, R2 = 0.8995145278326102
XGBOOST MAE = 5179.19694897768, RMSE = 6652.8252574026665, R2 = 0.8483265237633748
[경기] RF 학습완료, XGBoost 학습완료
랜덤포레스트 MAE = 19176.70703578125, RMSE = 24890.539832388007, R2 = 0.9794966978499573
XGBOOST MAE = 31294.27181026786, RMSE = 42907.1244639701, R2 = 0.9390724099291169
[경남] RF 학습완료, XGBoost 학습완료
랜덤포레스트 MAE = 7113.056995833938, RMSE = 9171.665034473786, R2 = 0.937699844239952
XGBOOST MAE = 5833.636006696428, RMSE = 7617.077660308823, R2 = 0.9570295868678165
[경북] RF 학습완료, XGBoost 학습완료
랜덤포레스트 MAE = 4978.135140439497, RMSE = 6895.548388096199, R2 = 0.9601035742573147
XGBOOST MAE = 8192.526259464285, RMSE = 10567.46619187987, R2 = 0.9063003210015655
[광주] RF 학습완료, XGBoost 학습완료
랜덤포레스트 MAE = 2777.973345238096, RMSE = 3502.065876766521, R2 = 0.9766342463724489
XGBOOST MAE = 3408.234549386161, RMSE = 4210.130631186108, R2 = 0.9662306801871662
[대구] RF 학습완료, XGBoost 학습완료
랜덤포레스트 MAE = 1

In [6]:
target = next((e for e in result_anls if e['region'] == '제주'), None)

print(target)
print(result_anls)

{'region': '제주', 'rf_mae': 328.8026034523804, 'rf_rmse': np.float64(526.0570758385575), 'rf_r2': 0.745694846713905, 'xg_mae': 348.54130998883926, 'xg_rmse': np.float64(553.9241878378683), 'xg_r2': 0.7180383216390569}
[{'region': '강원', 'rf_mae': 3961.823500512689, 'rf_rmse': np.float64(5415.055784047702), 'rf_r2': 0.8995145278326102, 'xg_mae': 5179.19694897768, 'xg_rmse': np.float64(6652.8252574026665), 'xg_r2': 0.8483265237633748}, {'region': '경기', 'rf_mae': 19176.70703578125, 'rf_rmse': np.float64(24890.539832388007), 'rf_r2': 0.9794966978499573, 'xg_mae': 31294.27181026786, 'xg_rmse': np.float64(42907.1244639701), 'xg_r2': 0.9390724099291169}, {'region': '경남', 'rf_mae': 7113.056995833938, 'rf_rmse': np.float64(9171.665034473786), 'rf_r2': 0.937699844239952, 'xg_mae': 5833.636006696428, 'xg_rmse': np.float64(7617.077660308823), 'xg_r2': 0.9570295868678165}, {'region': '경북', 'rf_mae': 4978.135140439497, 'rf_rmse': np.float64(6895.548388096199), 'rf_r2': 0.9601035742573147, 'xg_mae': 81